# VisDrone YOLOv8m Fine-tuning (Kaggle / Colab T4)

Mirrors `src/training/train.py`. Run on a **free T4 GPU** (Kaggle: *Settings → Accelerator → GPU T4*).

**Targets:** mAP@0.5 ≈ 0.87 in ~3 h. Report whatever the run actually produces.

## Before you run
1. Enable the GPU accelerator.
2. Add the VisDrone-DET dataset (Kaggle dataset search: *VisDrone*) or run the download cell.
3. Set your DagsHub MLflow URI + token in the **Config** cell below.

In [ ]:
# --- Install deps ---
!pip -q install ultralytics mlflow dagshub onnx onnxruntime

In [ ]:
# --- Verify GPU ---
import torch
assert torch.cuda.is_available(), 'No GPU! Enable the T4 accelerator.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# --- Config: FILL THESE IN ---
import os

# Your DagsHub MLflow URI, e.g. https://dagshub.com/<user>/<repo>.mlflow
MLFLOW_TRACKING_URI = ''  # <-- set me
os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI
os.environ['MLFLOW_TRACKING_USERNAME'] = ''  # <-- DagsHub username
os.environ['MLFLOW_TRACKING_PASSWORD'] = ''  # <-- DagsHub access token

DATA_YAML = '/kaggle/working/data.yaml'  # path to YOLO data.yaml
EPOCHS = 100
IMGSZ = 640
BATCH = 16
EXPERIMENT = 'visdrone-yolov8m'

## Data preparation

Clone this repo to reuse the conversion code, then download + convert VisDrone. If you attached a Kaggle VisDrone dataset, point `--config` paths at it instead.

In [ ]:
# --- Get the repo (replace with your fork URL) ---
# !git clone https://github.com/<you>/object-detection-tracking.git
# %cd object-detection-tracking
# !pip -q install -r requirements.txt
#
# !python -m src.data.download_visdrone --config configs/paths.yaml
# !python -m src.data.convert_visdrone  --config configs/paths.yaml --write-data-yaml
# !python -m src.data.validate_labels   --config configs/paths.yaml

In [ ]:
# --- Train with MLflow logging to DagsHub ---
import mlflow
from ultralytics import YOLO

if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT)

model = YOLO('yolov8m.pt')

def _log_epoch(trainer):
    if not MLFLOW_TRACKING_URI:
        return
    m = {k.replace('(', '').replace(')', ''): float(v) for k, v in trainer.metrics.items()}
    mlflow.log_metrics(m, step=trainer.epoch)

model.add_callback('on_fit_epoch_end', _log_epoch)

run = mlflow.start_run() if MLFLOW_TRACKING_URI else None
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=0, seed=42, cos_lr=True, close_mosaic=10,
    project='runs/train', name='yolov8m_visdrone',
)
print('Saved to:', results.save_dir)

In [ ]:
# --- Log final metrics + best.pt as artifacts, then close the run ---
from pathlib import Path

save_dir = Path(results.save_dir)
best = save_dir / 'weights' / 'best.pt'

if MLFLOW_TRACKING_URI:
    if model.metrics and model.metrics.results_dict:
        mlflow.log_metrics({k.replace('(', '').replace(')', ''): float(v)
                            for k, v in model.metrics.results_dict.items()
                            if isinstance(v, (int, float))})
    for p in ['confusion_matrix.png', 'results.png', 'PR_curve.png']:
        fp = save_dir / p
        if fp.exists():
            mlflow.log_artifact(str(fp), 'plots')
    if best.exists():
        mlflow.log_artifact(str(best), 'weights')
    mlflow.end_run()

print('best.pt:', best, '| exists:', best.exists())

## After training

Download `best.pt` (Kaggle: *Output* tab) and place it at `weights/best.pt` in the repo. Then resume locally with **Phase 4 (ONNX export + INT8 quantization)**. Paste the real mAP@0.5 / mAP@0.5:0.95 into the README results table.